In [2]:
import kagglehub

# This will download to a local cache on the Lightning instance
muhmagdy_valentini_noisy_path = kagglehub.dataset_download('muhmagdy/valentini-noisy')
print(f'Kaggle data is at: {muhmagdy_valentini_noisy_path}')

Kaggle data is at: /teamspace/studios/this_studio/.cache/kagglehub/datasets/muhmagdy/valentini-noisy/versions/3


In [3]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import librosa
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from natsort import natsorted
import random

In [4]:
clean_train_path_english = os.path.join(muhmagdy_valentini_noisy_path, 'clean_trainset_56spk_wav')
train_files = sorted(glob.glob(os.path.join(clean_train_path_english, "*.wav")))

clean_val_path_english = os.path.join(muhmagdy_valentini_noisy_path, 'clean_trainset_28spk_wav')
val_files = sorted(glob.glob(os.path.join(clean_val_path_english, "*.wav")))

print(f"Total Training Files: {len(train_files)}")
print(f"Total Val Files: {len(val_files)}")

Total Training Files: 23075
Total Val Files: 11572


In [5]:

base_path = "/teamspace/studios/this_studio/MeetVerse_DataSet"

clean_train_path_arabic = f"{base_path}/Clear Sound train"
clean_val_path_arabic = f"{base_path}/Clear Sound val"
noise_path = f"{base_path}/Noise/audio"
noise_people42 = f"{base_path}/Noise/People Noise "  

print(f"Arabic Train exists: {os.path.exists(clean_train_path_arabic)}")
print(f"Arabic Val exists: {os.path.exists(clean_val_path_arabic)}")
print(f"Noise exists: {os.path.exists(noise_path)}")
print(f"People Noise exists: {os.path.exists(noise_people42)}") 

Arabic Train exists: True
Arabic Val exists: True
Noise exists: True
People Noise exists: True


In [6]:
import os
import glob
import random
import soundfile as sf
import numpy as np
import torch
from torch.utils.data import Dataset

class SmartNoiseDataset(Dataset):
    def __init__(self, clean_dirs_list, general_noise_dir, babble_noise_dir, sr=16000, max_len=80000): #16000  256  0.016* 
        self.sr = sr
        self.max_len = max_len

        # Gather clean files recursively
        self.clear_files = []
        for d in clean_dirs_list:
            if os.path.exists(d):
                for ext in ('**/*.wav', '**/*.mp3'):
                    self.clear_files.extend(glob.glob(os.path.join(d, ext), recursive=True))

        random.shuffle(self.clear_files)

        # Gather noise sources
        self.general_noises = self._get_files(general_noise_dir)
        self.babble_noises = self._get_files(babble_noise_dir)

    def _get_files(self, directory):
        files = []
        for ext in ('**/*.wav', '**/*.mp3'):
            files.extend(glob.glob(os.path.join(directory, ext), recursive=True))
        return files

    def __len__(self):
        return len(self.clear_files)

    def _load_audio(self, path):
        # Using soundfile for high-speed I/O
        audio, fs = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        
        # Resample only if necessary
        if fs != self.sr:
            import librosa
            audio = librosa.resample(audio, orig_sr=fs, target_sr=self.sr)
        
        if len(audio) == 0:
            return np.zeros(self.max_len, dtype=np.float32)
            
        # Fix length: random crop for long files, zero-pad for short ones
        if len(audio) > self.max_len:
            start = random.randint(0, len(audio) - self.max_len)
            audio = audio[start:start + self.max_len]
        else:
            pad_width = self.max_len - len(audio)
            audio = np.pad(audio, (0, pad_width), mode='constant')
            
        return audio.astype(np.float32)

    def _smart_normalize(self, audio, target_db=-25):
        # RMS Normalization to ensure stable gradients during training
        rms = np.sqrt(np.mean(audio**2) + 1e-7)
        current_db = 20 * np.log10(rms + 1e-7)
        gain = 10**((target_db - current_db) / 20)
        
        norm_audio = audio * gain
        
        # Final safety check to prevent clipping above 1.0
        max_val = np.abs(norm_audio).max()
        if max_val > 0.99:
            norm_audio = norm_audio / (max_val + 1e-6)
            
        return norm_audio

    def _mix_with_snr(self, clean, noise, snr_range):
        snr_db = random.uniform(snr_range[0], snr_range[1])
        
        clean_rms = np.sqrt(np.mean(clean**2) + 1e-7)
        noise_rms = np.sqrt(np.mean(noise**2) + 1e-7)
        
        # Calculate scaling factor based on power ratio
        snr_linear = 10**(snr_db / 20.0)
        factor = clean_rms / (snr_linear * noise_rms)
        
        return clean + factor * noise

    def __getitem__(self, idx):
        c_path = self.clear_files[idx]
        fname = os.path.basename(c_path)

        try:
            # 1. Load Clean Audio
            clear = self._load_audio(c_path)
            
            # 2. Random Augmentation (Gain variation)
            clear = clear * random.uniform(0.6, 1.1)
            
            rand_val = random.random()
            
            # 3. Mixing Logic based on Cases
            if rand_val < 0.10:
                # Case 1: Almost clean (Sensor hiss only)
                noisy = clear + np.random.normal(0, 0.001, size=len(clear))
                
            elif rand_val < 0.45:
                # Case 2: General Noise (Stationary)
                n_gen = self._load_audio(random.choice(self.general_noises))
                noisy = self._mix_with_snr(clear, n_gen, snr_range=(5, 25))
                
            elif rand_val < 0.75:
                # Case 3: Babble Noise (Non-stationary)
                n_bab = self._load_audio(random.choice(self.babble_noises))
                noisy = self._mix_with_snr(clear, n_bab, snr_range=(0, 20))
                
            else:
                # Case 4: Heavy Mixed Noise (Stress test for the model)
                n_gen = self._load_audio(random.choice(self.general_noises))
                n_bab = self._load_audio(random.choice(self.babble_noises))
                combined_noise = (n_gen + n_bab) / 2
                noisy = self._mix_with_snr(clear, combined_noise, snr_range=(-5, 15))

            # 4. Balanced Normalization
            # We normalize BOTH noisy and clean to the same target level
            # This ensures the model learns mapping, not just volume scaling
            noisy = self._smart_normalize(noisy, target_db=-25)
            clear = self._smart_normalize(clear, target_db=-25)

            # 5. Final Clip and Casting
            noisy = np.clip(noisy, -1.0, 1.0)
            clear = np.clip(clear, -1.0, 1.0)
            
            return (torch.from_numpy(noisy).float(), 
                    torch.from_numpy(clear).float(), 
                    fname)
            
        except Exception:
            # Fallback for corrupted files
            return self.__getitem__(random.randint(0, len(self) - 1))

In [7]:
# Define all your training paths in one list
train_clean_paths = [
    clean_train_path_english,
    clean_train_path_arabic
]
# Define all your validation paths in another list
val_clean_paths = [
    clean_val_path_english,
    clean_val_path_arabic
]
# Create Datasets
train_ds = SmartNoiseDataset(train_clean_paths, noise_path, noise_people42)
val_ds = SmartNoiseDataset(val_clean_paths, noise_path, noise_people42)

# Create Loaders with Shuffle=True for maximum randomization
train_loader = DataLoader(train_ds, batch_size=6, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=6, shuffle=False, num_workers=0)

print(f"Total Unified Training Samples: {len(train_ds)}")
print(f"Total Unified validation Samples: {len(val_ds)}")

Total Unified Training Samples: 40445
Total Unified validation Samples: 18508


In [8]:
# from torch.utils.data import random_split, DataLoader

# # Define all your training paths in one list
# train_clean_paths = [
#     clean_train_path_english,
#     clean_train_path_arabic
# ]
# # Define all your validation paths in another list
# val_clean_paths = [
#     clean_val_path_english,
#     clean_val_path_arabic
# ]

# # Create Datasets
# train_ds = SmartNoiseDataset(train_clean_paths, noise_path, noise_people42)
# val_ds = SmartNoiseDataset(val_clean_paths, noise_path, noise_people42)

# train_third_len = len(train_ds) // 10
# train_rest_len = len(train_ds) - train_third_len

# val_third_len = len(val_ds) // 10
# val_rest_len = len(val_ds) - val_third_len

# train_ds_third, _ = random_split(train_ds, [train_third_len, train_rest_len])
# val_ds_third, _ = random_split(val_ds, [val_third_len, val_rest_len])

# # Create Loaders with Shuffle=True for maximum randomization
# train_loader = DataLoader(train_ds_third, batch_size=6, shuffle=True, num_workers=0)
# val_loader = DataLoader(val_ds_third, batch_size=6, shuffle=False, num_workers=0)

# print(f"Total Unified Training Samples (1/10): {len(train_ds_third)}")
# print(f"Total Unified validation Samples (1/10): {len(val_ds_third)}")

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        # Using Conv2d for channel-wise attention
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.pool(x)
        w = self.fc(w)
        return x * w

class ResSEBlock(nn.Module):
    def __init__(self, in_c, out_c, stride):
        super().__init__()
        self.act = nn.SiLU(inplace=True)
        
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        
        self.se = SEBlock(out_c)
        
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = self.se(out)
        
        out = out + identity
        out = self.act(out)
        
        return out

def up_block(in_c, out_c):
    # Bilinear upsampling to avoid checkerboard artifacts
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.SiLU(inplace=True)
    )

class BeastDenoiser(nn.Module):
    def __init__(self, base=48):
        super().__init__()
        self.base = base
        
        # Encoder path
        self.enc1 = ResSEBlock(2, base, stride=(2, 2))
        self.enc2 = ResSEBlock(base, base*2, stride=(2, 2))
        self.enc3 = ResSEBlock(base*2, base*4, stride=(2, 2))
        self.enc4 = ResSEBlock(base*4, base*8, stride=(2, 2))

        # BottleNeck with GRU for temporal dependencies
        # Flattened size depends on n_fft=512
        self.gru_input_size = (base*8) * 16 
        
        self.gru = nn.GRU(
            input_size=self.gru_input_size, 
            hidden_size=512,    # 256 128 
            num_layers=2, #1 
            batch_first=True,
            bidirectional=False
        )
        
        self.gru_fc = nn.Sequential(
            nn.Linear(512, self.gru_input_size),
            nn.SiLU(inplace=True)
        )

        # Decoder path with skip connections
        self.dec4 = up_block(base*16, base*4)
        self.dec3 = up_block(base*8, base*2)
        self.dec2 = up_block(base*4, base)
        
        self.mask_conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(base*2, 2, kernel_size=3, padding=1)
        )
        
        self.sig = nn.Sigmoid()

        # Audio parameters
        self.n_fft = 512
        self.hop_length = 128
        self.register_buffer("window", torch.hann_window(self.n_fft))

    def match_shape(self, x, target):
        # Ensure tensor dimensions match for skip connections
        diffY = target.size(2) - x.size(2)
        diffX = target.size(3) - x.size(3)
        return F.pad(x, (diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2))

    def forward(self, x_wav, h=None):
        # 1. Frequency Domain Transformation
        # Using center=True to handle signal boundaries properly
        spec = torch.stft(
            x_wav, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            window=self.window, 
            return_complex=True,
            center=True
        )
        spec_view = torch.view_as_real(spec).permute(0, 3, 1, 2)
        x = spec_view[:, :, :-1, :]

        # 2. Encoding path
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        # 3. RNN Processing
        b, c, f, t = e4.shape
        seq = e4.permute(0, 3, 1, 2).reshape(b, t, -1)
        rnn_out, next_h = self.gru(seq, h)
        rnn_out = self.gru_fc(rnn_out)
        rnn_out = rnn_out.reshape(b, t, c, f).permute(0, 2, 3, 1)

        # 4. Decoding & Shape Matching
        d4 = self.match_shape(self.dec4(torch.cat([rnn_out, e4], 1)), e3)
        d3 = self.match_shape(self.dec3(torch.cat([d4, e3], 1)), e2)
        d2 = self.match_shape(self.dec2(torch.cat([d3, e2], 1)), e1)

        # 5. Mask & Reconstruction
        mask = self.sig(self.match_shape(self.mask_conv(torch.cat([d2, e1], 1)), x))
        masked_spec = x * mask
        masked_spec = F.pad(masked_spec, (0, 0, 0, 1)).permute(0, 2, 3, 1).contiguous()
        
        out_wav = torch.istft(
            torch.complex(masked_spec[..., 0], masked_spec[..., 1]), 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            window=self.window, 
            length=x_wav.shape[-1]
        )
        return out_wav, next_h

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiScaleSpectralLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.fft_sizes   = [2048, 1024, 512, 256]
        self.hop_sizes   = [240,  120,  60,  30]
        self.win_lengths = [1200, 600,  240, 120]
        
        for i, win in enumerate(self.win_lengths):
            # register_buffer makes it part of the module state but not a parameter
            self.register_buffer(f"window_{i}", torch.hann_window(win))

    def forward(self, est, ref):
        total_loss = 0.0
        
        for i, (n_fft, hop, win) in enumerate(zip(self.fft_sizes, self.hop_sizes, self.win_lengths)):
            # Force the window to be on the same device as the input tensor (est)
            window = getattr(self, f"window_{i}").to(est.device)
            
            est_stft = torch.stft(est, n_fft, hop, win, window=window, return_complex=True, center=True)
            ref_stft = torch.stft(ref, n_fft, hop, win, window=window, return_complex=True, center=True)
            
            est_mag = torch.abs(est_stft).clamp(min=1e-7).pow(0.3)
            ref_mag = torch.abs(ref_stft).clamp(min=1e-7).pow(0.3)
            
            sc_loss = torch.norm(ref_mag - est_mag, p="fro") / (torch.norm(ref_mag, p="fro") + 1e-7)
            mag_loss = F.l1_loss(est_mag, ref_mag)
            
            log_ref = torch.log(ref_mag.clamp(min=1e-7))
            log_est = torch.log(est_mag.clamp(min=1e-7))
            log_loss = F.l1_loss(log_ref, log_est)
            
            total_loss += sc_loss + mag_loss + 1.0 * log_loss
            
        return total_loss / len(self.fft_sizes)

class MasterHybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.spectral_loss_fn = MultiScaleSpectralLoss()

    def forward(self, pred, target):
        # 1. Multi-scale Spectral Loss (with the Log-Magnitude update)
        spec_loss = self.spectral_loss_fn(pred, target)
        
        # 2. SI-SDR Loss calculation
        # Zero-mean for scale invariance
        target_zm = target - target.mean(dim=-1, keepdim=True)
        pred_zm = pred - pred.mean(dim=-1, keepdim=True)
        
        dot = (pred_zm * target_zm).sum(dim=-1, keepdim=True)
        target_energy = (target_zm**2).sum(dim=-1, keepdim=True) + 1e-8
        
        s_target = (dot / target_energy) * target_zm
        e_noise = pred_zm - s_target
        
        si_sdr = 10 * torch.log10((s_target**2).sum(dim=-1) / ((e_noise**2).sum(dim=-1) + 1e-8) + 1e-8)
        sisdr_loss = -si_sdr.mean()

        # THE BALANCED UPDATE:
        # 1.5 for natural quality, 1.2 for aggressive noise suppression
        total_loss = (1.5 * spec_loss) + (1.2 * sisdr_loss)
        
        return total_loss

In [11]:
def calculate_targets(clean, noisy, predicted):
    # Ensure all tensors are float32
    clean = clean.float()
    noisy = noisy.float()
    predicted = predicted.float()

    # 1. Calculate Signal-to-Distortion Ratio (SDR) as a base for Quality
    # This is more professional than just L2 differences
    target_energy = torch.sum(clean**2, dim=-1) + 1e-8
    error_energy = torch.sum((predicted - clean)**2, dim=-1) + 1e-8
    
    # Distortion Ratio (Strict)
    # If error is 20% of clean signal, quality is 80%
    distortion_ratio = torch.sqrt(error_energy / target_energy)
    quality_pct = (1.0 - distortion_ratio) * 100.0
    
    # 2. Noise Removal Accuracy
    # How much of the original noise was actually suppressed
    original_noise = noisy - clean
    original_noise_energy = torch.sum(original_noise**2, dim=-1) + 1e-8
    
    # We measure how much noise is left in the "silence" parts
    # A better proxy for "60% Noise Removed"
    noise_reduction = 1.0 - (error_energy / (original_noise_energy + 1e-8))
    noise_removal_pct = noise_reduction * 100.0

    # Final clamping and averaging
    quality_score = torch.clamp(quality_pct, 0.0, 100.0).mean().item()
    noise_score = torch.clamp(noise_removal_pct, 0.0, 100.0).mean().item()
    
    return quality_score, noise_score

In [12]:
# Upgrade the whole PyTorch ecosystem together to ensure compatibility
# !pip install --upgrade torch torchvision torchaudio torchmetrics pesq pystoi

In [13]:
# import torch
# import torch.nn as nn
# from tqdm import tqdm
# from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
# from torchmetrics.audio.stoi import ShortTimeObjectiveIntelligibility

# # 1. Setup Device and Professional Metrics
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using Device: {device}")

# # Initialize professional metrics for validation
# # 16000 Hz is required as discussed
# pesq_fn = PerceptualEvaluationSpeechQuality(16000, 'wb').to(device)
# stoi_fn = ShortTimeObjectiveIntelligibility(16000).to(device)

# # 2. Initialize Model and Criterion
# # Using base=48 for the balanced performance target
# model = BeastDenoiser(base=48).to(device)

# # Using the Hybrid Loss we optimized (1.5 Spectral + 1.2 SI-SDR)
# # Ensure your MasterHybridLoss class is defined above this
# criterion = MasterHybridLoss().to(device)

# optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)

# # Scheduler tracks avg_qual (max mode) to reduce LR if quality plateaus
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
# )

# history = {'train_loss': [], 'val_loss': [], 'quality': [], 'noise_rem': [], 'pesq': [], 'stoi': []}
# best_score = 0.0

# total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"BeastDenoiser Balanced is ready! Parameters: {total_params:,}")

Using Device: cuda
BeastDenoiser Balanced is ready! Parameters: 19,485,698


In [14]:
# EPOCHS = 3
# accumulation_steps = 4 # Adjusted for faster updates

# # 3. Training Loop
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss_epoch = 0.0
#     nan_batches = 0
#     optimizer.zero_grad() 
    
#     pbar_train = tqdm(train_loader, desc=f"[Train] Epoch {epoch+1}/{EPOCHS}")
#     for i, (noisy, clean, _) in enumerate(pbar_train):
#         noisy, clean = noisy.to(device), clean.to(device)
        
#         # Guard against zero-energy files
#         if noisy.abs().max() < 1e-6 or clean.abs().max() < 1e-6:
#             continue

#         # Forward pass with initial hidden state as None
#         # In training on random crops, we treat samples as independent
#         pred, _ = model(noisy, None)

#         # Shape matching (padding handling)
#         if pred.shape[-1] != clean.shape[-1]:
#             L = min(pred.shape[-1], clean.shape[-1])
#             pred, clean = pred[..., :L], clean[..., :L]

#         loss = criterion(pred, clean) / accumulation_steps

#         if not torch.isfinite(loss):
#             nan_batches += 1
#             optimizer.zero_grad() # Reset gradients if NaN occurs
#             continue

#         loss.backward()
        
#         # Gradient Accumulation
#         if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
#             nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
#             optimizer.step()
#             optimizer.zero_grad()

#         train_loss_epoch += loss.item() * accumulation_steps
#         pbar_train.set_postfix(loss=f"{loss.item()*accumulation_steps:.4f}")

#     avg_train = train_loss_epoch / max(len(train_loader) - nan_batches, 1)

#     # ================= Validation =================
#     model.eval()
#     val_loss_epoch, val_qual, val_nr, val_pesq, val_stoi = 0.0, 0.0, 0.0, 0.0, 0.0
#     val_count = 0

#     with torch.no_grad():
#         pbar_val = tqdm(val_loader, desc=f"[ Val ] Epoch {epoch+1}/{EPOCHS}")
#         for noisy, clean, _ in pbar_val:
#             noisy, clean = noisy.to(device), clean.to(device)
#             if noisy.abs().max() < 1e-6 or clean.abs().max() < 1e-6:
#                 continue

#             # Evaluation Forward
#             pred, _ = model(noisy, None)
            
#             if pred.shape[-1] != clean.shape[-1]:
#                 L = min(pred.shape[-1], clean.shape[-1])
#                 pred, clean = pred[..., :L], clean[..., :L]

#             loss_v = criterion(pred, clean)
#             val_loss_epoch += loss_v.item()

#             # A. Heuristic Targets (The 80/60 target check)
#             q, nr = calculate_targets(clean, noisy, pred)
#             val_qual += q
#             val_nr += nr
            
#             # B. Professional Metrics (The Comparison check)
#             # Metrics expect [Batch, Time] or [Time]
#             # try:
#                 # We calculate mean over batch for PESQ/STOI
#                 # p_val = pesq_fn(pred, clean).mean()
#                 # s_val = stoi_fn(pred, clean).mean()
#                 # val_pesq += p_val.item()
#                 # val_stoi += s_val.item()
#             # except:
#                 # pass # PESQ can fail on silence/heavy distortion
                
#             val_count += 1

#     # Average metrics
#     denom = max(val_count, 1)
#     avg_val = val_loss_epoch / denom
#     avg_qual = val_qual / denom
#     avg_nr = val_nr / denom
#     # avg_pesq = val_pesq / denom
#     # avg_stoi = val_stoi / denom

#     # Update Scheduler using Quality score
#     scheduler.step(avg_qual) 

#     # Save History
#     history['train_loss'].append(avg_train)
#     history['val_loss'].append(avg_val)
#     history['quality'].append(avg_qual)
#     history['noise_rem'].append(avg_nr)
#     # history['pesq'].append(avg_pesq)
#     # history['stoi'].append(avg_stoi)

#     current_lr = optimizer.param_groups[0]['lr']
    
#     # Check if we hit the global benchmark goals
#     status = "GOAL ACHIEVED" if (avg_qual > 80.0 and avg_nr > 60.0) else "In Progress"

#     print(f"\nEpoch {epoch+1:>3} Summary")
#     print(f"LR: {current_lr:.2e} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
#     print(f"Quality: {avg_qual:.2f}% | Noise Rem: {avg_nr:.2f}% | Status: {status}")
#     # print(f"Professional: PESQ: {avg_pesq:.2f} | STOI: {avg_stoi:.2f}")
#     print("-" * 60)

#     # Combined score for saving (60% Quality + 40% Noise Removal)
#     current_score = (avg_qual * 0.6) + (avg_nr * 0.4)
    
#     if current_score > best_score:
#         best_score = current_score
#         torch.save({
#             'epoch': epoch,
#             'model_state': model.state_dict(),
#             'optimizer_state': optimizer.state_dict(),
#             'best_score': best_score,
#             'history': history
#         }, "beast_balanced_best_update_full.pth")
#         print(f"Saved Best Model (Score={current_score:.2f})")

[ Val ] Epoch 1/3: 100%|██████████| 3085/3085 [08:22<00:00,  6.15it/s]



Epoch   1 Summary
LR: 3.00e-04 | Train Loss: -19.6794 | Val Loss: -21.0210
Quality: 81.07% | Noise Rem: 59.16% | Status: In Progress
------------------------------------------------------------
Saved Best Model (Score=72.30)


[ Val ] Epoch 2/3: 100%|██████████| 3085/3085 [08:18<00:00,  6.19it/s]



Epoch   2 Summary
LR: 3.00e-04 | Train Loss: -21.3647 | Val Loss: -21.8366
Quality: 81.89% | Noise Rem: 62.90% | Status: GOAL ACHIEVED
------------------------------------------------------------
Saved Best Model (Score=74.29)


[ Val ] Epoch 3/3: 100%|██████████| 3085/3085 [08:17<00:00,  6.20it/s]



Epoch   3 Summary
LR: 3.00e-04 | Train Loss: -21.9467 | Val Loss: -22.2033
Quality: 82.17% | Noise Rem: 65.20% | Status: GOAL ACHIEVED
------------------------------------------------------------
Saved Best Model (Score=75.38)


In [20]:
print('hello')

hello


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# الكتل الأساسية زي ما هي (SEBlock, ResSEBlock, up_block)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.pool(x)
        w = self.fc(w)
        return x * w

class ResSEBlock(nn.Module):
    def __init__(self, in_c, out_c, stride):
        super().__init__()
        self.act = nn.SiLU(inplace=True)
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SEBlock(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        out = out + identity
        return self.act(out)

def up_block(in_c, out_c):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.SiLU(inplace=True)
    )

# الموديل الرئيسي (قسمناه لجزأين داخلياً)
class BeastDenoiser(nn.Module):
    def __init__(self, base=48):
        super().__init__()
        self.base = base
        
        # Encoder
        self.enc1 = ResSEBlock(2, base, stride=(2, 2))
        self.enc2 = ResSEBlock(base, base*2, stride=(2, 2))
        self.enc3 = ResSEBlock(base*2, base*4, stride=(2, 2))
        self.enc4 = ResSEBlock(base*4, base*8, stride=(2, 2))

        # BottleNeck
        self.gru_input_size = (base*8) * 16 
        self.gru = nn.GRU(
            input_size=self.gru_input_size, 
            hidden_size=512, 
            num_layers=2, 
            batch_first=True,
            bidirectional=False
        )
        self.gru_fc = nn.Sequential(
            nn.Linear(512, self.gru_input_size),
            nn.SiLU(inplace=True)
        )

        # Decoder
        self.dec4 = up_block(base*16, base*4)
        self.dec3 = up_block(base*8, base*2)
        self.dec2 = up_block(base*4, base)
        
        self.mask_conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(base*2, 2, kernel_size=3, padding=1)
        )
        self.sig = nn.Sigmoid()

        # Audio parameters
        self.n_fft = 512
        self.hop_length = 128
        self.register_buffer("window", torch.hann_window(self.n_fft))

    def match_shape(self, x, target):
        diffY = target.size(2) - x.size(2)
        diffX = target.size(3) - x.size(3)
        return F.pad(x, (diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2))

    # 🔴 دالة جديدة: دي اللي هنصدرها لـ ONNX (بتاخد Spectrogram وترجع Mask)
    def network_forward(self, x, h=None):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b, c, f, t = e4.shape
        seq = e4.permute(0, 3, 1, 2).reshape(b, t, -1)
        rnn_out, next_h = self.gru(seq, h)
        rnn_out = self.gru_fc(rnn_out)
        rnn_out = rnn_out.reshape(b, t, c, f).permute(0, 2, 3, 1)

        d4 = self.match_shape(self.dec4(torch.cat([rnn_out, e4], 1)), e3)
        d3 = self.match_shape(self.dec3(torch.cat([d4, e3], 1)), e2)
        d2 = self.match_shape(self.dec2(torch.cat([d3, e2], 1)), e1)

        mask = self.sig(self.match_shape(self.mask_conv(torch.cat([d2, e1], 1)), x))
        return mask, next_h

    # الدالة الأصلية بتاعت التدريب
    def forward(self, x_wav, h=None):
        spec = torch.stft(
            x_wav, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            window=self.window, 
            return_complex=True,
            center=True
        )
        spec_view = torch.view_as_real(spec).permute(0, 3, 1, 2)
        x = spec_view[:, :, :-1, :]

        # استخدام الدالة المفصولة
        mask, next_h = self.network_forward(x, h)

        masked_spec = x * mask
        masked_spec = F.pad(masked_spec, (0, 0, 0, 1)).permute(0, 2, 3, 1).contiguous()
        
        out_wav = torch.istft(
            torch.complex(masked_spec[..., 0], masked_spec[..., 1]), 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            window=self.window, 
            length=x_wav.shape[-1]
        )
        return out_wav, next_h

# 🔴 غلاف (Wrapper) لتصدير الشبكة فقط لـ ONNX
class BeastDenoiserONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        mask, _ = self.model.network_forward(x)
        return mask

In [23]:
# 1. تهيئة الموديل وتحميل الأوزان
base_model = BeastDenoiser(base=48)
checkpoint = torch.load("beast_balanced_best_update_full.pth", map_location="cpu")
base_model.load_state_dict(checkpoint['model_state'])
base_model.eval()

# 2. تغليف الموديل ليكون جاهز للـ ONNX
onnx_model = BeastDenoiserONNXWrapper(base_model)
onnx_model.eval()

# 3. إدخال وهمي يمثل الـ Spectrogram مش الصوت الخام
# Shape: [Batch=1, Channels=2 (Real/Imag), FreqBins=256, TimeFrames=126]
dummy_spec = torch.randn(1, 2, 256, 0923093, dtype=torch.float32)

onnx_file_path = "beast_network_only.onnx"
print(f"Starting Clean ONNX export to {onnx_file_path}...")

torch.onnx.export(
    onnx_model,
    (dummy_spec,),
    onnx_file_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['spectrogram_input'],
    output_names=['mask_output'],
    
    # مسموح هنا بالـ Dynamic Axes عادي لأننا لغينا الـ STFT المعقد
    dynamic_axes={
        'spectrogram_input': {0: 'batch_size', 3: 'time_frames'},
        'mask_output': {0: 'batch_size', 3: 'time_frames'}
    },
    dynamo=False
)

print(f"✅ CLEAN ONNX Export Successful! Model saved at: {onnx_file_path}")

Starting Clean ONNX export to beast_network_only.onnx...


/tmp/ipykernel_1985/2298103124.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4571: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


✅ CLEAN ONNX Export Successful! Model saved at: beast_network_only.onnx


In [16]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 157.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 86.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 155.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [onnxscript]4 [onnxscript]


In [ ]:
plt.figure(figsize=(18, 5))

# Plot 1: Loss
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.legend()
plt.grid(True)

# Plot 2: Quality (Target > 85%)
plt.subplot(1, 3, 2)
plt.plot(history['quality'], color='green', marker='o', label='Actual Quality')
plt.axhline(y=85, color='r', linestyle='--', label='Target (85%)') # خط التارجت
plt.title('Audio Quality % (Target > 85%)')
plt.xlabel('Epoch')
plt.ylabel('Percentage %')
plt.legend()
plt.grid(True)

# Plot 3: Noise Removal (Target > 65%)
plt.subplot(1, 3, 3)
plt.plot(history['noise_rem'], color='blue', marker='x', label='Actual Noise Rem')
plt.axhline(y=65, color='r', linestyle='--', label='Target (65%)') # خط التارجت
plt.title('Noise Removal % (Target > 65%)')
plt.xlabel('Epoch')
plt.ylabel('Percentage %')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()